In [1]:
import os
import re
import pdfplumber

In [71]:
def get_pdf_paths():
    DATA_PATH = os.path.join("samples")
    pdf_paths = []
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Dir not found: {DATA_PATH}")

    for file in os.listdir(DATA_PATH):
        if file.endswith(".pdf"):
            pdf_paths.append(os.path.join(DATA_PATH, file))

    print(f"Znaleziono pliki: {pdf_paths}")
    return pdf_paths


In [72]:
pdf_paths = get_pdf_paths()
pdf_paths

Znaleziono pliki: ['samples\\karta_info_skargi_wnioski_sn_233c5c16.pdf', 'samples\\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf', 'samples\\OC_os_fiz_przy_EDU_Plus_2b489658.pdf']


['samples\\karta_info_skargi_wnioski_sn_233c5c16.pdf',
 'samples\\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf',
 'samples\\OC_os_fiz_przy_EDU_Plus_2b489658.pdf']

In [73]:
def extract_text_from_pdfs(pdf_paths):
    pdfs_texts = []
    for pdf_path in pdf_paths:
        full_pdf_text = ""

        print(f"Text extraction from: {os.path.basename(pdf_path)}\n")
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                temp_text = page.extract_text(layout=True)
                full_pdf_text += temp_text + "\n"
        pdfs_texts.append(" ".join(full_pdf_text.split()))
        print(f"Extracted text from {os.path.basename(pdf_path)}: {len(full_pdf_text)} characters\n")

    print("Check of first 300 characters of the first PDF text:\n")
    preview = pdfs_texts[0][:300] if pdfs_texts else ""
    print(preview)
    return pdfs_texts

In [74]:
pdfs_texts = extract_text_from_pdfs(pdf_paths)

Text extraction from: karta_info_skargi_wnioski_sn_233c5c16.pdf

Extracted text from karta_info_skargi_wnioski_sn_233c5c16.pdf: 11780 characters

Text extraction from: Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf

Extracted text from Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf: 214466 characters

Text extraction from: OC_os_fiz_przy_EDU_Plus_2b489658.pdf

Extracted text from OC_os_fiz_przy_EDU_Plus_2b489658.pdf: 156246 characters

Check of first 300 characters of the first PDF text:

KARTA INFORMACYJNA Poniżej znajdą Państwo niezbędne informacje dotyczące przetwarzania danych osobowych w ramach rozpatrywania wniosków o udostępnienie informacji publicznej przez organy Sądu Najwyższego. Administratorem danych osobowych przetwarzanych przy rozpatrywaniu wniosków o dostęp Kto jest a


In [ ]:
#For label studio
txt_folder = os.path.join("txt")
os.makedirs(txt_folder, exist_ok=True)

for pdf_path, pdf_text in zip(pdf_paths, pdfs_texts):
    txt_filename = os.path.splitext(os.path.basename(pdf_path))[0] + ".txt"
    txt_path = os.path.join(txt_folder, txt_filename)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(pdf_text)
    print(f"Saved extracted text to: {txt_path}")

Saved extracted text to: txt\karta_info_skargi_wnioski_sn_233c5c16.txt
Saved extracted text to: txt\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.txt
Saved extracted text to: txt\OC_os_fiz_przy_EDU_Plus_2b489658.txt


In [ ]:
#For label studio
import json
for pdf_path, pdf_text in zip(pdf_paths, pdfs_texts):
    txt_filename = os.path.splitext(os.path.basename(pdf_path))[0] + ".txt"
    json_data = {
        "data": {
            "text": pdf_text,
            "source": txt_filename
        }
    }
    json_filename = os.path.splitext(os.path.basename(pdf_path))[0] + ".json"
    json_path = os.path.join("txt_json", json_filename)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
    print(f"Saved JSON to: {json_path}")

Saved JSON to: txt_json\karta_info_skargi_wnioski_sn_233c5c16.json
Saved JSON to: txt_json\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.json
Saved JSON to: txt_json\OC_os_fiz_przy_EDU_Plus_2b489658.json


In [75]:
import re

def tokenize_text(pdfs_texts):
    token_pattern = re.compile(
    r"\d{2}-\d{3}|[\w]+|[^\w\s]",
    flags=re.UNICODE
    )
    pdfs_tokens = []
    for pdf_text in pdfs_texts:
        pdf_tokens = []

        for match in token_pattern.finditer(pdf_text):
            pdf_tokens.append({
                "token": match.group(),
                "start": match.start(),
                "end": match.end()
            })

        pdfs_tokens.append(pdf_tokens)

    return pdfs_tokens

In [79]:
pdfs_tokens = tokenize_text(pdfs_texts)
pdfs_tokens[0][310:410]  # Display the first 10 tokens of the first PDF

[{'token': '.', 'start': 2016, 'end': 2017},
 {'token': 'jeszcze', 'start': 2018, 'end': 2025},
 {'token': 'sposób', 'start': 2026, 'end': 2032},
 {'token': '?', 'start': 2032, 'end': 2033},
 {'token': 'Dane', 'start': 2034, 'end': 2038},
 {'token': 'gromadzone', 'start': 2039, 'end': 2049},
 {'token': 'w', 'start': 2050, 'end': 2051},
 {'token': 'ramach', 'start': 2052, 'end': 2058},
 {'token': 'rozpatrywania', 'start': 2059, 'end': 2072},
 {'token': 'wniosków', 'start': 2073, 'end': 2081},
 {'token': 'o', 'start': 2082, 'end': 2083},
 {'token': 'udostępnienie', 'start': 2084, 'end': 2097},
 {'token': 'informacji', 'start': 2098, 'end': 2108},
 {'token': 'publicznej', 'start': 2109, 'end': 2119},
 {'token': 'Komu', 'start': 2120, 'end': 2124},
 {'token': 'przekazywane', 'start': 2125, 'end': 2137},
 {'token': 'są', 'start': 2138, 'end': 2140},
 {'token': 'przez', 'start': 2141, 'end': 2146},
 {'token': 'organy', 'start': 2147, 'end': 2153},
 {'token': 'Sądu', 'start': 2154, 'end': 215

In [41]:
#W tej celli musimy określić chunk size, overlap i powinnismy dostać dict z 3 rzeczami 1. text, 2. start wdg znaków, 3. end wdg znaków. Jedendict jeden chunk, dicty w listcie [{text:STRING , start:INT , end: INT}]
def create_chunks(
    text: str,
    tokens: list[dict],
    chunk_size: int = 180,
    overlap: int = 30
) -> list[dict]:
    
    if not tokens:
        return []

    step = chunk_size - overlap
    chunks = []

    for token_start_index in range(0, len(tokens), step):
        token_end_index = min(
            token_start_index + chunk_size,
            len(tokens)
        )

        chunk_tokens = tokens[token_start_index:token_end_index]

        if not chunk_tokens:
            break

        chunk_start = chunk_tokens[0]["start"]
        chunk_end = chunk_tokens[-1]["end"]

        chunks.append({
            "text": text[chunk_start:chunk_end],
            "start": chunk_start,
            "end": chunk_end
        })

        if token_end_index >= len(tokens):
            break

    return chunks

In [47]:
chunks = create_chunks(
    text=pdfs_texts[0],
    tokens=pdfs_tokens[0],
    chunk_size=180,
    overlap=30
)

print(chunks[2])

{'text': 'w jakiś informacji profili preferencji osób, których dane dotyczą. jeszcze sposób? Dane gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji publicznej Komu przekazywane są przez organy Sądu Najwyższego mogą być przekazywane wyłącznie uprawnionym organom, moje dane osobowe? w tym sądom administracyjnym i organom ścigania. Czy moje dane są Dane osobowe gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji przekazywane poza Unię publicznej przez organy Sądu Najwyższego nie będą przekazywane poza teren Unii Europejską? Europejskiej. Dane osobowe gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji Przez jaki czas publicznej przez organy Sądu Najwyższego przechowywane są bezterminowo. Jest to bowiem przetwarzane są moje niezbędne z punktu widzenia możliwości wykazania przed właściwymi organami, że SN wykonał dane osobowe? ciążące na nim obowiązki. Jako administrator danych, SN zapewnia Państwu możliwość potwierdzenia, czy w Sądzie

In [24]:
def create_chunks_with_metadata(
    text: str,
    tokens: list[dict],
    chunk_size: int = 180,
    overlap: int = 30
) -> tuple[list[dict], list[str]]:
    """
    Dzieli tokeny na chunki z overlapem.

    Zwraca:
    1. chunks_metadata - chunki wraz z metadanymi i tokenami,
    2. chunks_text - same teksty chunków.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size musi być większy od 0.")

    if overlap < 0:
        raise ValueError("overlap nie może być ujemny.")

    if overlap >= chunk_size:
        raise ValueError("overlap musi być mniejszy niż chunk_size.")

    step = chunk_size - overlap

    chunks_metadata = []
    chunks_text = []

    for chunk_id, token_start_index in enumerate(
        range(0, len(tokens), step)
    ):
        token_end_index = min(
            token_start_index + chunk_size,
            len(tokens)
        )

        chunk_tokens = tokens[token_start_index:token_end_index]

        if not chunk_tokens:
            break

        text_start = chunk_tokens[0]["start"]
        text_end = chunk_tokens[-1]["end"]

        # Wycięcie tekstu bezpośrednio z oryginalnego dokumentu
        chunk_text = text[text_start:text_end]

        chunk_metadata = {
            "chunk_id": chunk_id,

            # Zakres tokenów w całej liście tokenów
            "token_start_index": token_start_index,
            "token_end_index": token_end_index,

            # Zakres znaków w oryginalnym tekście
            "text_start": text_start,
            "text_end": text_end,

            # Tokeny należące do chunka
            "tokens": chunk_tokens,

            # Gotowy tekst do przekazania dalej
            "text": chunk_text
        }

        chunks_metadata.append(chunk_metadata)
        chunks_text.append(chunk_text)

        if token_end_index >= len(tokens):
            break

    return chunks_metadata, chunks_text

In [33]:
chunks_metadata, chunks_text = create_chunks_with_metadata(
    text=pdfs_texts[0],
    tokens=pdfs_tokens[0],
    chunk_size=180,
    overlap=30
)
chunks_metadata[2], chunks_text[2]

({'chunk_id': 2,
  'token_start_index': 300,
  'token_end_index': 480,
  'text_start': 1951,
  'text_end': 3145,
  'tokens': [{'token': 'w', 'start': 1951, 'end': 1952},
   {'token': 'jakiś', 'start': 1953, 'end': 1958},
   {'token': 'informacji', 'start': 1959, 'end': 1969},
   {'token': 'profili', 'start': 1970, 'end': 1977},
   {'token': 'preferencji', 'start': 1978, 'end': 1989},
   {'token': 'osób', 'start': 1990, 'end': 1994},
   {'token': ',', 'start': 1994, 'end': 1995},
   {'token': 'których', 'start': 1996, 'end': 2003},
   {'token': 'dane', 'start': 2004, 'end': 2008},
   {'token': 'dotyczą', 'start': 2009, 'end': 2016},
   {'token': '.', 'start': 2016, 'end': 2017},
   {'token': 'jeszcze', 'start': 2018, 'end': 2025},
   {'token': 'sposób', 'start': 2026, 'end': 2032},
   {'token': '?', 'start': 2032, 'end': 2033},
   {'token': 'Dane', 'start': 2034, 'end': 2038},
   {'token': 'gromadzone', 'start': 2039, 'end': 2049},
   {'token': 'w', 'start': 2050, 'end': 2051},
   {'tok

In [ ]:
'''
LATER
    Requirements state that one token should be treated as one word, 
    and the tokenizer should be able to handle special characters and punctuation. 
    The tokenizer will be used to split the text into manageable chunks for further processing.
'''
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "allegro/herbert-base-cased", use_fast=True
)

model_max_length = tokenizer.model_max_length
if model_max_length is None or model_max_length > 100_000:
    model_max_length = 512

special_tokens_count = tokenizer.num_special_tokens_to_add(pair=False)
TOKENIZER_CHUNK_SIZE = max(1, model_max_length - special_tokens_count)

print(f"Tokenizer gotowy. Limit treści chunku: {TOKENIZER_CHUNK_SIZE} tokenów.")


Pobieranie i ładowanie tokenizera: allegro/herbert-base-cased...
Tokenizer gotowy. Limit treści chunku: 510 tokenów.


In [ ]:
'''
    LATER
'''
def split_text_into_chunks(text, tokenizer, max_tokens):
    parts = re.findall(r"\s*\S+", text)
    chunks = []
    current_parts = []
    current_token_count = 0

    for part in parts:
        part_token_count = len(tokenizer.tokenize(part))

        if current_parts and current_token_count + part_token_count > max_tokens:
            chunks.append("".join(current_parts))
            current_parts = [part]
            current_token_count = part_token_count
        else:
            current_parts.append(part)
            current_token_count += part_token_count

    if current_parts:
        chunks.append("".join(current_parts))

    return chunks

chunks = split_text_into_chunks(full_pdf_text, tokenizer, TOKENIZER_CHUNK_SIZE)
chunk_token_counts = [len(tokenizer.tokenize(chunk)) for chunk in chunks]

print(f"Utworzono {len(chunks)} chunków wyłącznie na potrzeby limitu tokenizera.")
print(f"Najdłuższy chunk: {max(chunk_token_counts, default=0)} / {TOKENIZER_CHUNK_SIZE} tokenów.")


Utworzono 53 chunków wyłącznie na potrzeby limitu tokenizera.
Najdłuższy chunk: 510 / 510 tokenów.


In [ ]:
'''
    LATER

'''
tokens = []

for chunk in chunks:
    tokens.extend(tokenizer.tokenize(chunk))

print(f"Tokeny całego PDF-a: {len(tokens)}")
print(tokens[:50])


Tokeny całego PDF-a: 23782
['Ubezpieczenie</w>', 'Odpowiedzi', 'alności</w>', 'cywilnej</w>', 'osób</w>', 'fizycznych</w>', 'w</w>', 'życiu</w>', 'prywatnym</w>', 'oraz</w>', 'nauczycieli</w>', 'i</w>', 'dyrektorów</w>', 'placówek</w>', 'oświatowych</w>', 'w</w>', 'ramach</w>', 'oferty</w>', 'E', 'D', 'U</w>', 'Plus</w>', 'Dokument</w>', 'zawierający</w>', 'informacje</w>', 'o</w>', 'produ', 'kcie</w>', 'ubezpieczeniowym</w>', 'Przedsiębiorstwo</w>', ':</w>', 'Inter', 'Ri', 'sk</w>', 'Towarzystwo</w>', 'Ubezpieczeń</w>', 'Spółka</w>', 'Ak', 'cyjna</w>', 'V', 'ien', 'na</w>', 'In', 'surance</w>', 'Group</w>', 'z</w>', 'siedzibą</w>', 'w</w>', 'Polsce</w>', ',</w>']


In [ ]:
'''
later
'''
tokenization_flow = {
    "Full_pdf_text": Full_pdf_text,
    "chunks": chunks,
    "tokens": tokens,
}

print("Gotowy przepływ:")
print(f"Full_pdf_text: {len(tokenization_flow['Full_pdf_text'])} znaków")
print(f"chunks: {len(tokenization_flow['chunks'])} elementów")
print(f"tokens: {len(tokenization_flow['tokens'])} elementów")


Gotowy przepływ:
Full_pdf_text: 155297 znaków
chunks: 53 elementów
tokens: 23782 elementów


In [ ]:
'''
    later
'''
assert isinstance(Full_pdf_text, str)
assert isinstance(chunks, list)
assert isinstance(tokens, list)
assert all(isinstance(chunk, str) for chunk in chunks)
assert all(isinstance(token, str) for token in tokens)
assert sum(chunk_token_counts) == len(tokens)

print("Walidacja struktury OK: Full_pdf_text -> chunks -> tokens")


Walidacja struktury OK: Full_pdf_text -> chunks -> tokens


In [ ]:
''' from transformers import pipeline

pipe = pipeline("token-classification", model="lexedit/herbert-polish-legal-ner")
results = pipe(chunks[2]['text'])
print(results)'''


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity': 'B-ORG', 'score': np.float32(0.67098844), 'index': 32, 'word': 'Sądu</w>', 'start': 203, 'end': 207}, {'entity': 'I-ORG', 'score': np.float32(0.6203742), 'index': 33, 'word': 'Najwyższego</w>', 'start': 208, 'end': 219}, {'entity': 'B-ORG', 'score': np.float32(0.65512294), 'index': 76, 'word': 'Sądu</w>', 'start': 488, 'end': 492}, {'entity': 'I-ORG', 'score': np.float32(0.5881896), 'index': 77, 'word': 'Najwyższego</w>', 'start': 493, 'end': 504}, {'entity': 'B-ORG', 'score': np.float32(0.64182657), 'index': 105, 'word': 'Sądu</w>', 'start': 692, 'end': 696}, {'entity': 'I-ORG', 'score': np.float32(0.60057205), 'index': 106, 'word': 'Najwyższego</w>', 'start': 697, 'end': 708}, {'entity': 'B-ORG', 'score': np.float32(0.66331273), 'index': 148, 'word': 'SN</w>', 'start': 933, 'end': 935}, {'entity': 'B-ORG', 'score': np.float32(0.7650448), 'index': 156, 'word': 'Sądzie</w>', 'start': 984, 'end': 990}, {'entity': 'I-ORG', 'score': np.float32(0.6893335), 'index': 157, 'word':

In [ ]:
#To co wyżej - klasyfikacja tokenów w chunku, z tym że obliczę od razu start i end w całym tekście, a nie w chunku. + walidacja na podstawie listy tokenów w całowym tekście czy lokalizacja sie zgadza
# Input: chunk(text, start, end)
from transformers import pipeline

def classify_tokens_in_chunk(chunk: dict) -> list[dict]:
    """
    Klasyfikuje tokeny w danym chunku i zwraca wyniki z uwzględnieniem start i end w całym tekście.

    Args:
        chunk (dict): Chunk zawierający 'text', 'start', 'end'.
        

    Returns:
        list[dict]: Lista wyników klasyfikacji z uwzględnieniem start i end w całym tekście.
    """
    
    pipe = pipeline("token-classification", model="lexedit/herbert-polish-legal-ner")
    results = pipe(chunk['text'])

    # Przekształcenie wyników, aby uwzględnić start i end w całym tekście
    for result in results:
        result['start'] += chunk['start']
        result['end'] += chunk['start']

    # Walidacja: sprawdzenie, czy tokeny w wynikach klasyfikacji zgadzają się z tokenami w całym tekście -> czy w pdfs_tokens[] na danej pozycji tokeny się zgadzają
    '''for result in results:
        token_text = result['word']
        token_start = result['start']
        token_end = result['end']

        # Znalezienie tokenu w pdfs_tokens, który odpowiada wynikowi klasyfikacji
        matching_tokens = [
            token for token in pdfs_tokens[0]  # Zakładamy, że analizujemy pierwszy PDF
            if token['start'] == token_start and token['end'] == token_end
        ]

        if not matching_tokens:
            raise ValueError(f"Token '{token_text}' z wyników klasyfikacji nie pasuje do żadnego tokenu w całym tekście.")'''

    return results
    



In [63]:
res = classify_tokens_in_chunk(chunks[2])
res

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity': 'B-ORG',
  'score': np.float32(0.67098844),
  'index': 32,
  'word': 'Sądu</w>',
  'start': 2154,
  'end': 2158},
 {'entity': 'I-ORG',
  'score': np.float32(0.6203742),
  'index': 33,
  'word': 'Najwyższego</w>',
  'start': 2159,
  'end': 2170},
 {'entity': 'B-ORG',
  'score': np.float32(0.65512294),
  'index': 76,
  'word': 'Sądu</w>',
  'start': 2439,
  'end': 2443},
 {'entity': 'I-ORG',
  'score': np.float32(0.5881896),
  'index': 77,
  'word': 'Najwyższego</w>',
  'start': 2444,
  'end': 2455},
 {'entity': 'B-ORG',
  'score': np.float32(0.64182657),
  'index': 105,
  'word': 'Sądu</w>',
  'start': 2643,
  'end': 2647},
 {'entity': 'I-ORG',
  'score': np.float32(0.60057205),
  'index': 106,
  'word': 'Najwyższego</w>',
  'start': 2648,
  'end': 2659},
 {'entity': 'B-ORG',
  'score': np.float32(0.66331273),
  'index': 148,
  'word': 'SN</w>',
  'start': 2884,
  'end': 2886},
 {'entity': 'B-ORG',
  'score': np.float32(0.7650448),
  'index': 156,
  'word': 'Sądzie</w>',
  '